# Airbnb - Data Visualizations Project

## Customer Approach

In [8]:
import pandas as pd
import numpy as np
import re # for the text count in amenities

import pandas as pd
# Load your CSV
df = pd.read_csv("../listings.csv")


In [9]:
# Availability message
df["Availability Message"] = df["has_availability"].apply(
lambda x: "This property is currently available"
if str(x).lower() in ["true", "t", "1"]
else "This property is currently unavailable"
)

# Instant Bookable message
df["Instant Bookable Message"] = df["instant_bookable"].apply(
lambda x: "It can be booked now."
if str(x).lower() == "t"
else "Apologies, this property cannot be booked instantly. The host will be in touch about your stay."
)

# Combined message in a presentable sentence (optional)
df["Property Message"] = df["Availability Message"] + ". " + df["Instant Bookable Message"]

# Transform area into county column. Remove unessary text and group dublins areas into Dublin. iloc or loc may work
df.loc[df['region_parent_name'] == 'Fingal', 'region_parent_name'] = 'Dublin'
df.rename(columns={'region_parent_name': 'County'}, inplace=True)

# Transform date columns to datetime
date_columns = [
    'host_since',
    'first_review',
    'last_review',
    'calendar_last_scraped',
    'last_scraped'
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')


In [10]:
# 1. Quality and Trust Metrics

# Overall Quality Score (weighted average, boost if Superhost)
review_cols = ['review_scores_rating', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_accuracy', 'review_scores_communication', 'review_scores_location', 'review_scores_value']
df['Overall_Quality_Score'] = df[review_cols].mean(axis=1)
df['Overall_Quality_Score'] *= np.where(df['host_is_superhost'].astype(str).str.lower().isin(['t']), 1.2, 1.0)

# Host Reliability Score (simple average, discuss with Guilherme if there could be additional weights to this)
reliability_cols = ['host_response_rate', 'host_acceptance_rate']
for col in reliability_cols:
    df[col] = pd.to_numeric(df[col].str.rstrip('%')) / 100
df['host_identity_verified_flag'] = df['host_identity_verified'].astype(str).str.lower().isin(['t']).astype(int)
df['Host_Reliability_Score'] = df[reliability_cols + ['host_identity_verified_flag']].mean(axis=1)

# Days until first review (days between host_since and first_review)
df['host_since'] = pd.to_datetime(df['host_since'])
df['first_review'] = pd.to_datetime(df['first_review']) #Review if you need this after you created your new date function
df['Review_Velocity_Days'] = (df['first_review'] - df['host_since']).dt.days

# Review Age (months since last_review) - How old is this last review?
df['calendar_last_scraped'] = pd.to_datetime(df['calendar_last_scraped'])
df['last_review'] = pd.to_datetime(df['last_review']) #Ensure date variable exists
df['Review_Age_Months'] = ((df['calendar_last_scraped'] - df['last_review']).dt.days / 30).round(1) # Set the number to an appropriate decimal place after subtraction calculation

In [11]:
# 2. Value and Pricing Metrics

# Price per Guest
df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float) # Do a lot of testing on this since it took a few tries to get this working
df['Price_per_Guest'] = df['price'] / df['accommodates'].replace(0, np.nan) # Simple division, replace 0 with NaN to avoid confusion price per guest outputs

# Price per Bedroom
df['Price_per_Bedroom'] = df['price'] / df['bedrooms'].replace(0, np.nan) # Same idea as price per guest code above

# Use a function to count amenities items in each string
def count_amenity_items(amenities): 
    # Finds all substrings within double quotes
    return len(re.findall(r'"(.*?)"', str(amenities))) # Len is used to count how many strings were found in each. Make sure that amenities is a string, read strings using regular expression function and use .? to catch anything between double quotes

df['Amenities_Item_Count'] = df['amenities'].apply(count_amenity_items) # If the above works, create a new column

# If the above function fails (count_amentiy_items), try this, it doesn't depend on regex (hopefully you fix the regex issue)
# def count_amenity_items(amenities):
    # Remove brackets and split by comma
    #items = [a.strip() for a in str(amenities)[1:-1].split(',') if a.strip()]
    #return len(items)

# Minimum Stay Flexibility # Basically we want to create parameters around minimum nights and then group them into categories
def min_stay_flex(row):
    if row['minimum_nights'] <= 2:
        return 'Highly Flexible'
    elif row['minimum_nights'] <= 7:
        return 'Moderately Flexible'
    else:
        return 'Long Stay Only'
df['Minimum_Stay_Flexibility'] = df.apply(min_stay_flex, axis=1) # Add new column to dateset with this


In [12]:
# 3. Availability and Occupancy Metrics

# Occupancy Rate (90-Day) - Usig error -> coerce to handle errors without throwing code errors as some of these may be blank or non-numeric
df['availability_90'] = pd.to_numeric(df['availability_90'], errors='coerce')
df['Occupancy_Rate_90d'] = 1 - (df['availability_90'] / 90) # Try to build a proportion here, 1 - (available days / total days in period). Test if this works otherwise add plenty of ### to this and review it later

# Occupancy Rate (365-Day) - Usig error -> coerce to handle errors without throwing code errors as some of these may be blank or non-numeric
df['availability_365'] = pd.to_numeric(df['availability_365'], errors='coerce')
df['Occupancy_Rate_365d'] = 1 - (df['availability_365'] / 365) # Try to build a proportion here, 1 - (available days / total days in period). Test if this works otherwise add plenty of ### to this and review it later

# Listing Age (Years)
df['last_scraped'] = pd.to_datetime(df['last_scraped'], errors='coerce')
df['Listing_Age_Years'] = ((df['last_scraped'] - df['host_since']).dt.days / 365).round(2)  # Is there a better name for this variable?

# Revenue Density (Divide the estimated revenue in last 365 days by the number of people a listing can accommodate)
df['Revenue_Density'] = df['estimated_revenue_l365d'] / df['accommodates'].replace(0, np.nan)

# export to excel file...
df.to_csv("Ireland_Airbnb_Listing__Detailed__Customer.csv", index=False)